In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random

C:\Users\hp\AppData\Local\Temp\ipykernel_6920\1225448160.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

In [4]:
search_tool = DuckDuckGoSearchRun(region="us-en")

@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.

    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num

        elif operation == "sub":
            result = first_num - second_num

        elif operation == "mul":
            result = first_num * second_num

        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num

        else:
            return {"error": f"unsupported operation '{operation}'"}

        return {
            "first_num": first_num,
            "second_num": second_num,
            "operation": operation,
            "result": result
        }

    except Exception as e:
        return {"error": str(e)}

@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA')
    using Alpha Vantage with API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=IOJ6HICD8MSWY1Q7"
    r = requests.get(url)
    return r.json()
    

In [5]:
tools = [get_stock_price, search_tool, calculator]

# Make the LLM tool_aware
llm_with_tools = llm.bind_tools(tools)

In [6]:
 # state
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [7]:
#graph nodes 
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages" : [response]}

tool_node = ToolNode(tools) #Execute tool calls

In [8]:
# graph structure 
graph = StateGraph(ChatState)
graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)



In [9]:
graph.add_edge(START, 'chat_node')

# If the LLM asked for a tool, go to ToolNode; else finish
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge("tools", "chat_node")



In [10]:
chatbot = graph.compile()

In [11]:
# Regular chat
out = chatbot.invoke({"messages": [HumanMessage(content="What is 2*3")]})

print(out["messages"][-1].content[0]["text"])

2 * 3 = 6


In [12]:
print(chatbot.get_graph().draw_ascii())

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
        +-----------+         
        | chat_node |         
        +-----------+         
          .         .         
        ..           ..       
       .               .      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


In [13]:
# Chat requiring tool
out = chatbot.invoke({
    "messages": [
        HumanMessage(
            content="What is the stock price of Apple? How much would it cost to purchase 50 shares?"
        )
    ]
})

print(out["messages"][-1].content[0]["text"])

The current stock price of Apple (AAPL) is $332.27. 

To purchase 50 shares at this price, it would cost $16,613.50.


In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

print("Tracing:", os.getenv("LANGSMITH_TRACING"))
print("Project:", os.getenv("LANGSMITH_PROJECT"))
print("API key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))

Tracing: true
Project: LangGraph-Gemini
API key loaded: True
